<a href="https://colab.research.google.com/github/hsamala688/Space_Weather_Predictor/blob/main/tec_sfno_test_eval_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TEC SFNO — Test Set Evaluation

Loads the trained checkpoint (`best_model.pt`) and evaluates it on the held-out **test** split — never used for training or checkpoint selection.

## Cell 1 — Mount Drive & extract test data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, tarfile, json, os

DRIVE_TARBALL_PATH = "/content/drive/MyDrive/tec_data/New_Data?.gz"

shutil.copy(DRIVE_TARBALL_PATH, "New_Data.gz")
with tarfile.open("New_Data.gz", "r:gz") as tf:
    tf.extractall(".")

with open('data/falisha_windows_gl23x45/metadata.json') as f:
    metadata = json.load(f)
print(json.dumps(metadata, indent=2))


Mounted at /content/drive


/tmp/ipykernel_966/746402913.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(".")


{
  "format": "disk-backed normalized numpy arrays",
  "lmax": 22,
  "nlat": 23,
  "nlon": 45,
  "input_steps": 6,
  "target_steps": 3,
  "omni_features": [
    "b_magnitude",
    "by_gsm",
    "bz_gsm",
    "flow_speed",
    "proton_density",
    "kp_3hour"
  ],
  "train_end_year": 2019,
  "val_end_year": 2022,
  "normalization": {
    "tec_mean": 3.3914478323013326,
    "tec_std": 13.534793709634751,
    "omni_mean": [
      5.646234406753004,
      0.014151381683464744,
      -0.017162349380776525,
      430.86786592450795,
      6.1975210757779,
      1.777482807364012
    ],
    "omni_std": [
      3.02975265134192,
      3.889634799002429,
      3.2673296119045006,
      101.93470430892599,
      4.972585334453262,
      1.344105176850843
    ],
    "computed_from": "train tec_input and train omni_input only",
    "applied_to": "tec_input, omni_input, and target arrays"
  },
  "splits": {
    "train": 110124,
    "val": 25406,
    "test": 26270
  },
  "residual_definition": "dTEC

## Cell 2 — Load data pipeline code

Upload `falisha_dataset.py` via the Colab file browser first, then run this to place it correctly.

In [2]:
os.makedirs('data_pull', exist_ok=True)
os.rename('falisha_dataset.py', 'data_pull/falisha_dataset.py')


## Cell 3 — Install libraries & imports

In [3]:
!pip install -q torch-harmonics wandb

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torch_harmonics import RealSHT, InverseRealSHT
from torch_harmonics.quadrature import legendre_gauss_weights

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.6/537.6 kB 35.4 MB/s eta 0:00:00
Using device: cuda


## Cell 4 — Config

Must match the values used during training exactly.

In [4]:
CONFIG = {
    'H':     23,
    'W':     45,
    'l_max': 23,
    'm_max': 23,

    'n_timesteps':     6,
    'n_omni_features': 6,   # b_magnitude, by_gsm, bz_gsm, flow_speed, proton_density, kp_3hour
    'n_horizons':      3,

    'gru_hidden_size': 128,
    'sfno_channels':   [64, 128, 128, 64],
    'n_sfno_blocks':   4,

    'batch_size':      64,
    'learning_rate':   1e-4,   # unused for eval, load_checkpoint needs an optimizer object to exist

    'data_root': 'data/falisha_windows_gl23x45',
}

print('Config loaded.')
print(f"Grid: {CONFIG['H']}x{CONFIG['W']} | L_max: {CONFIG['l_max']}")


Config loaded.
Grid: 23x45 | L_max: 23


## Cell 5 — Build test DataLoader

In [3]:
from data_pull.falisha_dataset import make_falisha_dataloader

test_loader = make_falisha_dataloader(
    root=CONFIG['data_root'],
    split="test",
    batch_size=CONFIG['batch_size'],
    shuffle=False,
)

_batch = next(iter(test_loader))
print(_batch["tec_input"].shape, _batch["omni_input"].shape, _batch["target"].shape)


ModuleNotFoundError: No module named 'data_pull'

## Cell 6 — Model architecture

Must exactly match what was trained (same `SFNOBlock` with the Gibbs window, same `SphericalConvGRUCell`, same `TECSFNOModel`) or `load_state_dict` will fail.

In [6]:
class SFNOBlock(nn.Module):
    def __init__(self, in_channels, out_channels, H, W, l_max, m_max):
        super().__init__()
        self.sht  = RealSHT(H, W, lmax=l_max, mmax=m_max, grid='legendre-gauss')
        self.isht = InverseRealSHT(H, W, lmax=l_max, mmax=m_max, grid='legendre-gauss')

        self.filter_re = nn.Parameter(
            torch.randn(out_channels, in_channels, l_max, m_max) * 0.02
        )
        self.filter_im = nn.Parameter(
            torch.randn(out_channels, in_channels, l_max, m_max) * 0.02
        )

        l_idx = torch.arange(l_max, dtype=torch.float32)
        sigma = 1.0 - l_idx / l_max
        self.register_buffer('sigma_l', sigma.view(1, 1, -1, 1))

        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.act       = nn.GELU()

    def forward(self, x):
        residual   = self.pointwise(x)
        x_hat      = self.sht(x)
        W_c        = torch.complex(self.filter_re, self.filter_im)
        x_filtered = torch.einsum('bilm,oilm->bolm', x_hat, W_c)
        x_filtered = x_filtered * self.sigma_l
        x_out      = self.isht(x_filtered)
        return self.act(x_out + residual)


class SphericalConvGRUCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, H, W, l_max, m_max):
        super().__init__()
        self.hidden_channels = hidden_channels
        combined = input_channels + hidden_channels

        self.reset_gate  = SFNOBlock(combined, hidden_channels, H, W, l_max, m_max)
        self.update_gate = SFNOBlock(combined, hidden_channels, H, W, l_max, m_max)
        self.candidate   = SFNOBlock(combined, hidden_channels, H, W, l_max, m_max)

    def forward(self, x_t, h_prev):
        cat_xh  = torch.cat([x_t, h_prev], dim=1)
        r       = torch.sigmoid(self.reset_gate(cat_xh))
        z       = torch.sigmoid(self.update_gate(cat_xh))
        cat_xrh = torch.cat([x_t, r * h_prev], dim=1)
        h_cand  = torch.tanh(self.candidate(cat_xrh))
        return (1 - z) * h_prev + z * h_cand


class TECSFNOModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        H        = config['H']
        W        = config['W']
        l_max    = config['l_max']
        m_max    = config['m_max']
        gru_h    = config['gru_hidden_size']
        channels = config['sfno_channels']
        n_omni   = config['n_omni_features']
        n_horiz  = config['n_horizons']
        self.channels = channels

        self.omni_gru     = nn.GRU(n_omni, gru_h, batch_first=True)
        self.context_proj = nn.Linear(gru_h, channels[0])

        self.sph_gru = SphericalConvGRUCell(
            input_channels=1, hidden_channels=channels[0],
            H=H, W=W, l_max=l_max, m_max=m_max
        )

        ins  = [channels[0], channels[1], channels[2], channels[3]]
        outs = [channels[1], channels[2], channels[3], channels[3]]
        self.sfno_blocks = nn.ModuleList([
            SFNOBlock(ins[i], outs[i], H, W, l_max, m_max)
            for i in range(4)
        ])

        self.output_head = nn.Conv2d(channels[-1], n_horiz, kernel_size=1)

    def forward(self, tec_input, omni_input):
        B, T, H, W = tec_input.shape

        _, h_n  = self.omni_gru(omni_input)
        context = h_n.squeeze(0)
        h_init  = self.context_proj(context)
        h = h_init.unsqueeze(-1).unsqueeze(-1).expand(
            B, self.channels[0], H, W
        ).contiguous()

        for t in range(T):
            x_t = tec_input[:, t].unsqueeze(1)
            h   = self.sph_gru(x_t, h)

        x = h
        for block in self.sfno_blocks:
            x = block(x)

        return self.output_head(x)


print('Model classes defined.')


Model classes defined.


## Cell 7 — Load trained checkpoint

In [7]:
DRIVE_CHECKPOINT_PATH = "/content/drive/MyDrive/tec_data/best_model.pt"
shutil.copy(DRIVE_CHECKPOINT_PATH, "best_model.pt")

model     = TECSFNOModel(CONFIG).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])  # needed only so load_checkpoint has something to load into

def load_checkpoint(model, optimizer, path='best_model.pt'):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f}')
    return ckpt['epoch'], ckpt['val_loss']

epoch, val_loss = load_checkpoint(model, optimizer)
model.eval()


Loaded checkpoint from epoch 16, val_loss=0.0013


TECSFNOModel(
  (omni_gru): GRU(6, 128, batch_first=True)
  (context_proj): Linear(in_features=128, out_features=64, bias=True)
  (sph_gru): SphericalConvGRUCell(
    (reset_gate): SFNOBlock(
      (sht): RealSHT(
        nlat=23, nlon=45,
         lmax=23, mmax=23,
         grid=legendre-gauss, csphase=True
      )
      (isht): InverseRealSHT(
        nlat=23, nlon=45,
         lmax=23, mmax=23,
         grid=legendre-gauss, csphase=True
      )
      (pointwise): Conv2d(65, 64, kernel_size=(1, 1), stride=(1, 1))
      (act): GELU(approximate='none')
    )
    (update_gate): SFNOBlock(
      (sht): RealSHT(
        nlat=23, nlon=45,
         lmax=23, mmax=23,
         grid=legendre-gauss, csphase=True
      )
      (isht): InverseRealSHT(
        nlat=23, nlon=45,
         lmax=23, mmax=23,
         grid=legendre-gauss, csphase=True
      )
      (pointwise): Conv2d(65, 64, kernel_size=(1, 1), stride=(1, 1))
      (act): GELU(approximate='none')
    )
    (candidate): SFNOBlock(
    

## Cell 8 — De-normalization & GL area weights

Model output is a normalized residual, not real TEC — convert back before scoring anything.

In [8]:
norm = metadata['normalization']

TEC_MEAN = torch.tensor(norm['tec_mean'], dtype=torch.float32, device=DEVICE)
TEC_STD  = torch.tensor(norm['tec_std'],  dtype=torch.float32, device=DEVICE)

def denormalize_tec(x):
    """x: normalized residual dTEC -> real dTEC units."""
    return x * TEC_STD + TEC_MEAN

def make_gl_weights(H, device):
    _, w = legendre_gauss_weights(H)
    w    = w.to(torch.float32).to(device)
    w    = w / w.sum()
    return w.view(1, 1, H, 1)

gl_weights = make_gl_weights(CONFIG['H'], DEVICE)
print(f'TEC mean/std: {TEC_MEAN.item():.4f}, {TEC_STD.item():.4f}')
print(f'GL weights shape: {gl_weights.shape}')


TEC mean/std: 3.3914, 13.5348
GL weights shape: torch.Size([1, 1, 23, 1])


## Cell 9 — Evaluation metric functions

RMSE + spatial correlation (previously unimplemented Cell 13 deliverables).

In [9]:
def spatial_correlation(pred_real, target_real):
    """Pearson correlation between predicted and true TEC maps, per sample, per horizon."""
    B, n_horiz, H, W = pred_real.shape
    p = pred_real.reshape(B, n_horiz, -1)
    t = target_real.reshape(B, n_horiz, -1)
    p_c = p - p.mean(dim=-1, keepdim=True)
    t_c = t - t.mean(dim=-1, keepdim=True)
    num = (p_c * t_c).sum(dim=-1)
    den = torch.sqrt((p_c ** 2).sum(dim=-1) * (t_c ** 2).sum(dim=-1) + 1e-8)
    corr = num / den
    return corr.mean(dim=0)   # [n_horiz]


def compute_metrics(pred_real, target_real, gl_weights, horizon_names=['t+1h', 't+2h', 't+3h']):
    metrics = {}
    sq_err  = (pred_real - target_real) ** 2 * gl_weights
    corr    = spatial_correlation(pred_real, target_real)
    for i, name in enumerate(horizon_names):
        metrics[f'rmse_{name}'] = sq_err[:, i].mean().sqrt().item()
        metrics[f'corr_{name}'] = corr[i].item()
    return metrics


class PersistenceBaseline:
    """Repeat last observed TEC -- the bar the model needs to beat."""
    def predict(self, tec_input):
        return tec_input[:, -1:].expand(-1, 3, -1, -1)

print('Metric functions defined.')


Metric functions defined.


## Cell 10 — Storm/quiet threshold from test-set Kp

Uses the top quartile of test-set Kp (last input timestep) as the storm cutoff.

In [10]:
all_kp = torch.cat([batch["omni_input"][:, -1, 5] for batch in test_loader])
storm_threshold = torch.quantile(all_kp, 0.75).item()

print(f'Test-set Kp range: [{all_kp.min():.3f}, {all_kp.max():.3f}], mean={all_kp.mean():.3f}')
print(f'Storm threshold (top quartile, normalized Kp): {storm_threshold:.3f}')


Test-set Kp range: [-1.322, 5.373], mean=0.269
Storm threshold (top quartile, normalized Kp): 0.910


## Cell 11 — Run full evaluation

Overall RMSE/correlation, skill score vs. persistence, and storm-time vs. quiet-time (reported separately, never averaged).

In [11]:
@torch.no_grad()
def evaluate_test_set(model, loader, gl_weights, denorm_fn, storm_threshold):
    persistence = PersistenceBaseline()
    all_pred, all_target, all_persist, all_kp = [], [], [], []
    n_batches = len(loader)

    for i, batch in enumerate(loader):
        tec_input  = batch["tec_input"].to(DEVICE)
        omni_input = batch["omni_input"].to(DEVICE)
        target     = batch["target"].to(DEVICE)
        kp         = omni_input[:, -1, 5].contiguous()

        pred = model(tec_input, omni_input)

        all_pred.append(denorm_fn(pred).cpu())
        all_target.append(denorm_fn(target).cpu())
        all_persist.append(denorm_fn(persistence.predict(tec_input)).cpu())
        all_kp.append(kp.cpu())

        if i % 50 == 0:
            print(f'  batch {i}/{n_batches}')

    all_pred    = torch.cat(all_pred, dim=0)
    all_target  = torch.cat(all_target, dim=0)
    all_persist = torch.cat(all_persist, dim=0)
    all_kp      = torch.cat(all_kp, dim=0)
    gl_w        = gl_weights.cpu()

    overall  = compute_metrics(all_pred, all_target, gl_w)
    persist  = compute_metrics(all_persist, all_target, gl_w)
    skill    = {f'skill_{h}': 1 - (overall[f'rmse_{h}'] / persist[f'rmse_{h}']) for h in ['t+1h', 't+2h', 't+3h']}

    storm_mask = all_kp >= storm_threshold
    quiet_mask = ~storm_mask
    storm = compute_metrics(all_pred[storm_mask], all_target[storm_mask], gl_w) if storm_mask.sum() > 0 else {}
    quiet = compute_metrics(all_pred[quiet_mask], all_target[quiet_mask], gl_w) if quiet_mask.sum() > 0 else {}

    return {
        'overall': overall, 'persistence': persist, 'skill_score': skill,
        'storm': storm, 'quiet': quiet,
        'n_storm_samples': int(storm_mask.sum()), 'n_quiet_samples': int(quiet_mask.sum()),
    }


results = evaluate_test_set(model, test_loader, gl_weights, denormalize_tec, storm_threshold)

for section in ['overall', 'persistence', 'skill_score']:
    print(f'=== {section} ===')
    for k, v in results[section].items():
        print(f'  {k}: {v:.4f}')
    print()

print(f'=== storm-time (n={results["n_storm_samples"]}) ===')
for k, v in results['storm'].items():
    print(f'  {k}: {v:.4f}')
print()

print(f'=== quiet-time (n={results["n_quiet_samples"]}) ===')
for k, v in results['quiet'].items():
    print(f'  {k}: {v:.4f}')


  batch 0/411
  batch 50/411
  batch 100/411
  batch 150/411
  batch 200/411
  batch 250/411
  batch 300/411
  batch 350/411
  batch 400/411
=== overall ===
  rmse_t+1h: 0.6526
  corr_t+1h: 0.9906
  rmse_t+2h: 0.9866
  corr_t+2h: 0.9785
  rmse_t+3h: 1.2655
  corr_t+3h: 0.9659

=== persistence ===
  rmse_t+1h: 1.5585
  corr_t+1h: 0.9497
  rmse_t+2h: 2.9641
  corr_t+2h: 0.8186
  rmse_t+3h: 4.2005
  corr_t+3h: 0.6361

=== skill_score ===
  skill_t+1h: 0.5813
  skill_t+2h: 0.6671
  skill_t+3h: 0.6987

=== storm-time (n=7173) ===
  rmse_t+1h: 0.7109
  corr_t+1h: 0.9895
  rmse_t+2h: 1.0860
  corr_t+2h: 0.9760
  rmse_t+3h: 1.3858
  corr_t+3h: 0.9622

=== quiet-time (n=19097) ===
  rmse_t+1h: 0.6292
  corr_t+1h: 0.9910
  rmse_t+2h: 0.9466
  corr_t+2h: 0.9795
  rmse_t+3h: 1.2173
  corr_t+3h: 0.9673


In [6]:
from data_pull.falisha_dataset import make_falisha_dataloader

train_loader = make_falisha_dataloader(root=CONFIG['data_root'], split="train", batch_size=1, shuffle=False)
test_loader  = make_falisha_dataloader(root=CONFIG['data_root'], split="test",  batch_size=1, shuffle=False)

# Pull whatever timestamp/index field the dataset exposes per sample
train_times = [batch["timestamp"] for batch in train_loader]  # adjust field name to whatever falisha_dataset.py actually returns
test_times  = [batch["timestamp"] for batch in test_loader]

print("Train range:", min(train_times), "to", max(train_times))
print("Test range:", min(test_times), "to", max(test_times))
print("Overlap:", set(train_times) & set(test_times))

Train range: tensor([946713600]) to tensor([1577833200])
Test range: tensor([1672556400]) to tensor([1767222000])
Overlap: set()


In [7]:
import datetime

def to_date(ts):
    """Convert a unix timestamp tensor/int to a readable UTC date string."""
    ts_int = int(ts.item()) if hasattr(ts, "item") else int(ts)
    return datetime.datetime.fromtimestamp(ts_int, datetime.UTC).strftime("%Y-%m-%d %H:%M")

train_min, train_max = min(train_times), max(train_times)
test_min,  test_max  = min(test_times),  max(test_times)

print("Train range:", to_date(train_min), "to", to_date(train_max))
print("Test range:",  to_date(test_min),  "to", to_date(test_max))
print("Overlap:", set(train_times) & set(test_times))

gap_days = (test_min - train_max).item() / 86400 if hasattr(test_min, "item") else (test_min - train_max) / 86400
print(f"\nGap between train_end and test_start: {gap_days:.1f} days")


Train range: 2000-01-01 08:00 to 2019-12-31 23:00
Test range: 2023-01-01 07:00 to 2025-12-31 23:00
Overlap: set()

Gap between train_end and test_start: 1096.3 days
